# Parameters

In [1]:
from pathlib import Path

import numpy as np

from simplex_qp import load_problem_config, partition_from_config, save_partition_metadata

config = load_problem_config()
np.random.seed(config.seed)

n = config.n
k = config.k
m = config.block_size
sizes = list(config.block_sizes)

save_path = config.data_folder
save_path.mkdir(parents=True, exist_ok=True)
config.results_folder.mkdir(parents=True, exist_ok=True)
config.summaries_folder.mkdir(parents=True, exist_ok=True)
save_folder = str(save_path) + "/"

partition = partition_from_config(config)
save_partition_metadata(
    save_path,
    partition,
    extra={"source": "problem_config", "problem_config": "config/problem_config.json"},
)

print(f"n: {n}, k: {k}, m: {m}, sizes: {sizes}")
print(f"Folder: {save_folder}")


n: 1000, k: 100, m: 10, sizes: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
Folder: private/dim_n1000_k100/


# Generate U

In [2]:
from scipy.stats import ortho_group
U = ortho_group.rvs(n)

# print the matrix
np.set_printoptions(precision=3, suppress=True, linewidth=200)
print(U)

[[ 0.016 -0.004  0.02  ... -0.07  -0.002 -0.   ]
 [ 0.044  0.031  0.006 ... -0.019  0.041 -0.009]
 [-0.021 -0.005 -0.026 ... -0.046  0.01  -0.004]
 ...
 [ 0.007  0.    -0.031 ...  0.004  0.053  0.024]
 [-0.02  -0.033  0.074 ... -0.03   0.007 -0.031]
 [-0.044 -0.021  0.015 ...  0.004  0.011 -0.017]]


# well defined case

## Diag well

In [3]:
import numpy as np

# create a diagonal matrix with n random values on the diagonal greater or equal to 0 in the interval [0,10]
D_well = np.diag(np.random.rand(n) * 10)

# get one random index to set to 0
random_index = np.random.randint(0, n)
D_well[random_index, random_index] = 0

# count how many indexes are 0
num_zeros = np.sum(np.diag(D_well) == 0)
print(f"Number of zeros in the diagonal: {num_zeros}")

#print the matrix
np.set_printoptions(precision=3, suppress=True, linewidth=200)
print(D_well)

Number of zeros in the diagonal: 1
[[2.115 0.    0.    ... 0.    0.    0.   ]
 [0.    9.017 0.    ... 0.    0.    0.   ]
 [0.    0.    5.604 ... 0.    0.    0.   ]
 ...
 [0.    0.    0.    ... 1.268 0.    0.   ]
 [0.    0.    0.    ... 0.    0.252 0.   ]
 [0.    0.    0.    ... 0.    0.    6.618]]


## Q_well

In [4]:
Q_well = U @ D_well @ U.T

# Ill defined case

## Diag ill

In [5]:
import numpy as np

# create a diagonal matrix with n random values on the diagonal greater or equal to 0 in the interval [0,10e4]
D_ill = np.diag(np.random.rand(n) * 10e4)

# get one random index to set to 0
random_index = np.random.randint(0, n)
D_ill[random_index, random_index] = 0

# count how many indexes are 0
num_zeros = np.sum(np.diag(D_ill) == 0)
print(f"Number of zeros in the diagonal: {num_zeros}")

#print the matrix
np.set_printoptions(precision=3, suppress=True, linewidth=200)
print(D_ill)

Number of zeros in the diagonal: 1
[[78861.7       0.        0.    ...     0.        0.        0.   ]
 [    0.    46532.55      0.    ...     0.        0.        0.   ]
 [    0.        0.    65596.183 ...     0.        0.        0.   ]
 ...
 [    0.        0.        0.    ... 79263.456     0.        0.   ]
 [    0.        0.        0.    ...     0.    13423.168     0.   ]
 [    0.        0.        0.    ...     0.        0.    86773.875]]


## Q_ill

In [6]:
Q_ill = U @ D_ill @ U.T

# Save everything

In [7]:
# save the matrices U,D_ill, D_well,Q_ill,Q_well in npz file compressed
np.savez_compressed(save_folder + "matrices.npz", U=U, D_ill=D_ill, D_well=D_well, Q_ill=Q_ill, Q_well=Q_well)